# Qwen2.5:7B + RAG — clone dự án và đánh giá qa_test

Notebook chỉ clone dự án và gọi mã trong `src/`. Chọn **Runtime > Change runtime type > GPU** khi chạy mô hình. Chạy các ô từ trên xuống.

**Trước khi chạy:** mã nguồn mới phải có trên GitHub; đặt `GIT_REF` đúng nhánh/tag/commit. Notebook không lấy được các thay đổi chỉ nằm trên máy cá nhân. Không đặt token GitHub trong URL hoặc lưu trong notebook. Dự án riêng tư cần cấu hình xác thực Git của phiên Colab trước.

## 1. Clone dự án
Nếu đổi phiên bản mã sau khi đã import module, khởi động lại phiên Python trước khi tiếp tục.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

REPO_URL = "https://github.com/qtamtensor05/ViGovBot.git"
GIT_REF = "codex/feat-add-setup-rag-database-management"  # Nhánh hiện tại; đổi main sau khi merge hoặc dùng commit cố định.
REPO_DIR = Path("/content/ViGovBot")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("REPO_DIR đang trỏ tới dự án khác; chọn thư mục mới")
    changes = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
    if changes.strip():
        raise RuntimeError("Có thay đổi trong checkout Colab; lưu lại hoặc chọn REPO_DIR mới trước khi cập nhật")
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Commit đang chạy:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2. Cài thư viện
Cài từ file requirements của đúng phiên bản dự án vừa clone.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-rag.txt"], check=True)

## 3. Kết nối Google Drive
Chọn tài khoản chứa dữ liệu và cấp quyền khi Colab yêu cầu.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Tạo cấu hình cho phiên Colab
Mã pipeline nằm trong `src/`; file `rag_config.yaml` chứa cấu hình mặc định.
Ô này chỉ thay đường dẫn/tài nguyên rồi lưu bản cấu hình riêng bên ngoài checkout.

Chuẩn bị trên Drive: `RAG_Data/unified.zip` chứa hai file unified, và `RAG_Data/qa_test/dataset.jsonl`.
Có thể dùng thư mục unified đã giải nén hoặc `qa_test.zip` chứa `dataset.jsonl` trong thư mục con.
Metadata 6 GB sẽ được đọc theo luồng sang SQLite trên ổ đĩa Colab, không nạp toàn bộ vào RAM.

In [ ]:
import yaml
config = yaml.safe_load((REPO_DIR / "rag_config.yaml").read_text(encoding="utf-8"))
config["data"].update({
    "unified_source": "/content/drive/MyDrive/RAG_Data/unified.zip",
    "dataset_path": "/content/drive/MyDrive/RAG_Data/qa_test/dataset.jsonl",
    "dataset_zip": None,  # Nếu dùng ZIP: đặt dataset_path=None, dataset_zip=".../qa_test.zip".
    "cache_dir": "/content/tthc_rag_cache",
    "output_dir": "/content/drive/MyDrive/RAG_Data/qwen2_5_7b_rag_results",
})
config["embedding"]["device"] = "cpu"  # Dành VRAM cho Qwen; đổi cuda nếu đủ VRAM.
config["embedding"]["revision"] = None  # Cùng revision đã dùng khi tạo vector.
config["retrieval"]["top_k"] = 5
config["evaluation"]["max_cases"] = None  # None = toàn bộ 570 câu; thử 10 câu với output_dir riêng.
config["evaluation"]["smoke_test_n"] = 5
config["evaluation"]["bertscore_device"] = "cpu"
config["evaluation"]["bertscore_batch_size"] = 1
CONFIG_PATH = Path("/content/rag_colab.yaml")
CONFIG_PATH.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(CONFIG_PATH.read_text(encoding="utf-8"))

## 5. Chuẩn bị Ollama
Hàm trong dự án cài Ollama khi cần, khởi động dịch vụ và tải Qwen2.5:7B. Lần đầu cần tải trọng số mô hình.

In [ ]:
from src.utils.colab_runtime import ensure_ollama
ensure_ollama(config["llm"]["model"], config["llm"]["ollama_url"])

## 6. Chạy pipeline từ src
`run` thực hiện: chuẩn bị SQLite → nạp BGE-M3/FAISS → thử 5 câu → đánh giá → giải phóng mô hình → tính điểm và xuất báo cáo.
Giữ cùng cấu hình và thư mục kết quả để tiếp tục các câu chưa thành công nếu Colab bị ngắt.
Nếu đổi code, dữ liệu hoặc tham số, dùng `output_dir` mới.

Có thể đổi `COMMAND` thành `prepare`, `smoke`, `evaluate`, hoặc `report` để chạy riêng từng bước.
`report` dùng kết quả đã lưu, không cần nạp Qwen/BGE-M3. BERTScore mặc định CPU có thể chạy lâu.

In [ ]:
COMMAND = "run"
subprocess.run([sys.executable, "-m", "src.rag", "--config", str(CONFIG_PATH), COMMAND],
               cwd=REPO_DIR, check=True)

## 7. Xem kết quả trên Drive
Báo cáo gồm Exact Match, Accuracy chuẩn hóa, BLEU-4, ROUGE, BERTScore như baseline; thêm nguồn truy hồi và thời gian. Accuracy là khớp chuỗi, không phải đánh giá đầy đủ độ đúng pháp lý. Tổng hợp ghi rõ số câu còn thiếu nếu có lỗi.

In [ ]:
import json
import pandas as pd
output = Path(config["data"]["output_dir"])
summary_path = output / "summary.json"
if summary_path.exists():
    display(pd.DataFrame([json.loads(summary_path.read_text(encoding="utf-8"))]))
print("Thư mục kết quả:", output)
print("Các file:", [p.name for p in output.iterdir()] if output.exists() else [])